In [22]:
import pymupdf

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.chat_models import init_chat_model

# rag
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Loader
from langchain_community.document_loaders import PyPDFLoader

# Parser
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

# 임베딩 모델
from langchain_ollama import OllamaEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings

# 벡터 스토어
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS

from langchain_classic.chains.query_constructor.base import AttributeInfo

# selfquery
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor,EmbeddingsFilter, DocumentCompressorPipeline, CrossEncoderReranker
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever

# Reranking
from langchain_cohere import CohereRerank


In [21]:
# env 파일 읽어오기

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [3]:
# 모델 선언

llm = init_chat_model(model="gpt-6-luna",reasoning={"effort":"low"})
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

#### 기본적인 RAG 방식 문제점
- 중복 문서 문제
- 키워드 의존성
- 구조적 질의 불가 : ex) 2023년 이후 계약 중 금액이 1억 이상인 제품
- 노이즈 청크 : 관련성이 낮은 청크가 LLM에게 전달되어 환각 유발
- 긴 컨텍스트 문제 : 불필요한 정보가 많아질수록 LLM 집중도 저하

In [ ]:
### 1. Chunking 전략 - chunk_size, chunk_overlap, 임베딩 모델

#### 2. MMR(Maximal Marginal Relevance) Retriever
- 기본 Similarity Search 에서 가장 유사한 문서 반환 => 유사한 내용의 문서가 나옴
- 관련성 + 다양성 
- fetch_k = 20 추출 => 이미 고른 문서와 덜 비슷한 것 K 개 찾기


In [5]:
def create_chunks_form_pdf(pdf_path):
        
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(docs)

    chunks = [chunk for chunk in chunks if chunk.page_content.strip()]
    return chunks


def create_vectorstore(chunks, embedding, collection_name,persist_directory="./db/chroma_db"):
    return Chroma.from_documents(documents=chunks, 
                                     embedding=embedding,
                                     persist_directory=persist_directory,
                                     collection_name=collection_name)
# 다양성 값 : 0 ~ 1(일반적 값: 0.5)
def create_retriever(vector_store,search_type="similarity",k=4,fetch_k=20,lambda_mult=0.5):

    kwargs = {"k":k}

    if search_type == "mmr":
        kwargs['fetch_k'] = fetch_k
        kwargs['lambda_mult'] = lambda_mult
        

    return vector_store.as_retriever(search_type=search_type, search_kwargs=kwargs)


def print_retrieved_docs(title, retriever, query):
    docs = retriever.invoke(query)

    print("\n"+"="*50)
    print(title)
    print("="*50)

    for i, doc in enumerate(docs):
        print(f"page={doc.metadata.get('page')}")
        print(doc.page_content)

In [6]:
chunks = create_chunks_form_pdf("./data/제주관광가이드.pdf")
vectorstore = create_vectorstore(chunks=chunks, embedding=embeddings,collection_name="jeju1")

# mmr retriever
mmr_retriver = create_retriever(vector_store=vectorstore,search_type="mmr",k=3,fetch_k=10)
# 기존
sim_retriver = create_retriever(vector_store=vectorstore,k=3)

# query
query = "제주도에서 먹어 볼만한 음식은?"

print_retrieved_docs("MMR SEARCH RESULT",mmr_retriver,query)
print_retrieved_docs("Similarity SEARH RESULT",sim_retriver,query)


KeyboardInterrupt: 

#### 3. SelfQuery Retriever
- 자연어 질의를 분석하여 `시멘틱 검색 쿼리`와 `메타 데이터 필터`를 LLM이 자동으로 생성

In [8]:
docs = [
    Document(
        page_content="삼성전자 제품 마케팅 직무입니다.",
        metadata = {"year":2025,"department":"marketing"}
    ),
    Document(
        page_content="AI 연구 개발 직무입니다.",
        metadata = {"year":2023,"department":"ai"}
    ),
    Document(
        page_content="백엔드 개발 직무입니다.",
        metadata = {"year":2025,"department":"developer"}
    ),
]

metadata_field_info = [
    AttributeInfo(
        name='year',
        description='문서 작성 연도',
        type='integer'
    ),
    AttributeInfo(
        name='department',
        description='직무 부서',
        type='string'
    ) 
]

document_content_description = "회사 내부 문서 및 정책 자료"

In [9]:
# ChromaTranslator() => 2024년 이후, 이전, => eq, gte ,le.. => llm 모델에 따라 충분할수도 있고 아닐수도있음
from langchain_core.structured_query import Comparison,Comparator
class SafeChromaTranslator(ChromaTranslator):
    def visit_comparison(self, comparison: Comparison):
        range_ops = (Comparator.GT, Comparator.GTE, Comparator.LT, Comparator.LTE)
        v = comparison.value

        if comparison.comparator in range_ops and isinstance(v, str):
            try:
                comparison.value = int(v)
            except ValueError:
                comparison.value = float(v)
        return super().visit_comparison(comparison)

vectorstore = create_vectorstore(chunks=docs, embedding=embeddings, collection_name="self_query1")

selfquery_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    enable_limit=True,
    structured_query_translator=SafeChromaTranslator()
)

In [11]:
query = "2024년 이전 ai 부서 직무를 찾아줘"

docs = selfquery_retriever.invoke(query)

for doc in docs:
    print(doc.page_content, doc.metadata)

AI 연구 개발 직무입니다. {'year': 2023, 'department': 'ai'}
AI 연구 개발 직무입니다. {'department': 'ai', 'year': 2023}
AI 연구 개발 직무입니다. {'department': 'ai', 'year': 2023}


In [13]:
# 화장품 상품의 설명과 메타데이터 생성
docs = [
    Document(
        page_content="수분 가득한 히알루론산 세럼으로 피부 속 깊은 곳까지 수분을 공급합니다.",
        metadata={"year": 2024, "category": "스킨케어", "user_rating": 4},
    ),
    Document(
        page_content="24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.",
        metadata={"year": 2023, "category": "메이크업", "user_rating": 3},
    ),
    Document(
        page_content="식물성 성분으로 만든 저자극 클렌징 오일, 메이크업과 노폐물을 부드럽게 제거합니다.",
        metadata={"year": 2023, "category": "클렌징", "user_rating": 5},
    ),
    Document(
        page_content="비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.",
        metadata={"year": 2023, "category": "스킨케어", "user_rating": 2},
    ),
    Document(
        page_content="롱래스팅 립스틱, 선명한 발색과 촉촉한 사용감으로 하루종일 편안하게 사용 가능합니다.",
        metadata={"year": 2024, "category": "메이크업", "user_rating": 4},
    ),
    Document(
        page_content="자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.",
        metadata={"year": 2025, "category": "선케어", "user_rating": 5},
    ),
]




# 메타데이터 필드 정보 생성
metadata_field_info = [
    AttributeInfo(
        name="category",
        description="The category of the cosmetic product. One of ['스킨케어', '메이크업', '클렌징', '선케어']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the cosmetic product was released",
        type="integer",
    ),
    AttributeInfo(
        name="user_rating",
        description="A user rating for the cosmetic product, ranging from 1 to 5",
        type="integer",
    ),
]


document_contents="Brief summary of a cosmetic product"



In [14]:
vectorstore = create_vectorstore(chunks=docs, embedding=embeddings, collection_name="self_query2")

selfquery_retriever2 = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_contents,
    metadata_field_info=metadata_field_info,
    verbose=True, # 디버깅용
    enable_limit=True, # 'n 개 알려줘' 같은 형태 지원 여부
    structured_query_translator=SafeChromaTranslator()
)

In [15]:
selfquery_retriever2.invoke("2024년 이후로 평점이 4 이상인 제품을 추천해줘")

[Document(id='b09cee99-7a7c-4d4b-a8b9-f42eaaaae489', metadata={'category': '선케어', 'user_rating': 5, 'year': 2025}, page_content='자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.')]

In [16]:
selfquery_retriever2.invoke("평점이 4 이상인 제품을 추천해줘")

[Document(id='0e709af1-9951-4ad7-9804-133c97864aa1', metadata={'category': '메이크업', 'year': 2024, 'user_rating': 4}, page_content='롱래스팅 립스틱, 선명한 발색과 촉촉한 사용감으로 하루종일 편안하게 사용 가능합니다.'),
 Document(id='b5d4e3b5-37a3-4d6a-8f90-453594cdadf6', metadata={'user_rating': 4, 'year': 2024, 'category': '스킨케어'}, page_content='수분 가득한 히알루론산 세럼으로 피부 속 깊은 곳까지 수분을 공급합니다.'),
 Document(id='b09cee99-7a7c-4d4b-a8b9-f42eaaaae489', metadata={'category': '선케어', 'year': 2025, 'user_rating': 5}, page_content='자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.'),
 Document(id='6c962f0e-40c4-409d-bd4c-81b040a12823', metadata={'category': '클렌징', 'user_rating': 5, 'year': 2023}, page_content='식물성 성분으로 만든 저자극 클렌징 오일, 메이크업과 노폐물을 부드럽게 제거합니다.')]

In [17]:
selfquery_retriever2.invoke("카테고리가 선케어인 상품을 추천해줘")

[Document(id='b09cee99-7a7c-4d4b-a8b9-f42eaaaae489', metadata={'user_rating': 5, 'category': '선케어', 'year': 2025}, page_content='자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.')]

#### 4. BM25 Hybrid Retriever
- 유사도 기반 => 정확한 제품명, 코드, 버전, 고유 명사 등의 키워드 매칭에는 약함
- 유사도 + 키워드 검색(TF-IDF) 알고리즘

In [ ]:
import json

with open("./data/samsung_dx_ocr.json","r",encoding="utf-8") as f:
    pages = json.load(f)

docs = []
for page in pages:
    docs.append(
        Document(
            page_content=page['content'],
            metadata={"page":page['page'],"source":"samsung_dx_ocr"}
        )
    )
# 자르기
splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
# 청크
chunks = splitter.split_documents(docs)

vectorstore = create_vectorstore(chunks=chunks, embedding=embeddings,collection_name="samsung_dx_ocr")

# 의미 검색
sim_retriver = create_retriever(vectorstore)

# 키워드기반
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4

hybrid_retriever = EnsembleRetriever(retrievers=[sim_retriver, bm25_retriever],weights=[0.3,0.7])

#query = "제품/서비스 마케팅"
query = "시스템 소프트웨어 자격 요견에 운영체제 개념도 포함되어 있어?"
print_retrieved_docs(title="Hybrid",retriever=hybrid_retriever,query=query)
print_retrieved_docs(title="Sim",retriever=sim_retriver,query=query)



Hybrid
page=7
자격요건 Requirements
컴퓨터 전기 전자, 기계 로봇공학 등 관련 전공올 하신 분 운영체제 기본 개념에 대한 이해도틀 보유하신 분 요구사항울 분석하여 소프트웨어들 구조적으로 설계 및 구현하는 역량울 보유하신 분 다양한 분야의 엔지니어와 적극적으로 소통하여 협업할 수 짓는 역량울 보유하신 분
우대사항 Preferences
CIC++ 기반 시스템 프로그래망 역량울 보유하신 분 Linux 기반 임베디드 시스템 개발 경험올 보유하신 분(Kernel, Device Driver, BSP 등) CAN EtherCAT, UDP 등 하드웨어 통신 프로토록 활용 경험 보유하신 분 ROS2 기반 로보텍스 프로적트 수행 경험올 보유하신 분 Git 기반 협업 및 CICD 환경에서의 개발 경험올 보유하신 분
커리어 비전 Career Vision
page=7
미래로봇추진단(서울 근무) S/W개발 시스템 소프트웨어
포지션 소개 Job Overview
휴머노이드 로봇의 실시간 제어 시스템 및 소프트웨어 플렉품올 개발하는 직무입니다: 운영체제 환경 구성, 디바이스 드라이버, 실시간 제어 프레임위크 등 로봇 동작의 핵심 기반이 되는 소프트웨어름 설계 및 개발하다, 하드웨어 설계 조직 및 Al 연구 조직과 긴밀히 현업합니다:
수행업무 Job Details
휴머노이드 로봇의 실시간 제어 프레임위크 설계하고 개발합니다: 모터; 센서 등 로봇 하드웨어 제어름 위한 디바이스 드라이버 및 하드웨어 추상화 계층올 개발합니다. 로봇 제어용 운영체제(Linux 기반 실시간 OS 등) 환경올 구성하고 시스템 성능올 최적화합니다: 유관 부서와 협업하여 보행 조작 등 Al 기능과 제어 시스템 간 연동 미들웨어름 개발합니다:
자격요건 Requirements
page=7
커리어 비전 Career Vision
휴머노이드 로봇의 초기 개발 단계부터 참여하여 핵심 개발자로 성장할 수 있습니다: 실시간 제어; 시스템 아키택처 , 로봇 미들웨어 등 다양한 경험올 통해 로보텍스 시스템 SW 전

In [19]:
print_retrieved_docs(title="Sim",retriever=sim_retriver,query=query)


Sim
page=18
MX사업부(수원 근무) 마켓팅 제품) 서비스 마게팅
포지션 소개 Job Overview
디바이스(Smartphone Watch Buds Tablet 등) 경험 및 핵심 차별화지 기획/상품화하고 고객과의 효과적인 커류니키이선 전락으로 경영성과 및 브랜드가치틀 제고하는 직무입니다:
수행업무 Job Details
브랜드 관리 신제품 런칭 등 마켓팅 커류니키이선 전락올 수립하고 실행합니다: 온라인-디지털 마켓팅 전락올 수립하고 캠페인올 개발합니다. 런칭 이번트 전시, 홍보 영상 손렌즈틀 기획합니다: 시장 소비자 경쟁사 기술 분석올 통한 기회 요인올 발굴하고 제품 전락올 수립합니다:
자격요건 Requirements
page=17
DA사업부(수원 근무) 마켓팅 제품) 서비스 마게팅
포지션 소개 Job Overview
고객 시장 경쟁사 기술 트랜드에 대한 이해틀 바탕으로 신규 제품올 기획하고 관리하는 직무로 고객 여정 팀구름 통해 인사이트트 찾아 새로운 제품과 서비스 영역올 발굴합니다:
수행업무 Job Details
기술의 발전 방향과 사회적 변화틀 예측하여 미래의 소비자가 요구하는 제품과 서비스의 큰섭트트 발굴합니다: 시장 및 소비자 트랜드틀 분석하여 제품의 차별화 기능 및 제품 포지서님 전락올 수립합니다. 판매 지역별 최적의 제품 라인업 전락울 수립하여 런칭부터 관리까지 수행합니다: 글로벌 파트너사 연계 서비스틀 기획하고 현업흘 진행합니다:
자격요건 Requirements
창의적인 사고와 신규 기술 제품에 대한 관심도가 높으신 분 Global 이문화에 대한 이해도가 높으신 분 팀위크와 커류니키이선 역량울 보유하신 분
우대사항 Preferences
page=16
영상디스플레이사업부(수원 근무) 마켓팅 제품) 서비스 마켓팅
포지션 소개 Job Overview
다양한 고객 인사이트트 활용하여 제품서비스트 차별화하고 새로운 고객 경험올 발줄하다, 고객의 가치름 높이는 제품| 서비스트 기획합니다:
수행업무 Job Details
시장소비자/ 경쟁사/기술 분

#### 5. Reranker
- 후보군 가져와서 질문-청크 쌍을 정밀하게 채점
- Bi-Encoder : 쿼리, 문서 독립적으로 처리 (대규모 검색에 우수함)
- Cross-Encoder : 쿼리 + 문서 쌍으로 처리 (느릴 수 있으나 정확도 매우 높음)

- 1차 검색 : Bi-Encoder 처리 => 2차 검색 : Cross-Encoder

- Cross-Encoder => `Cohere Reranker`

In [27]:
base_retriever = create_retriever(vector_store=vectorstore,k=20)

rerank = CohereRerank(model="rerank-v4.0-pro",top_n=5)

# 질문과 관련된 부분만 추출/압축 후 전달
compression_retriever = ContextualCompressionRetriever(base_compressor=rerank,base_retriever=base_retriever)

query="원격 근무 정책은 어떻게 되나요?"

#print_retrieved_docs(title="Rerank + Sim",retriever=compression_retriever, query=query)

docs = compression_retriever.invoke(query)
for doc in docs:
    print(f"rele_score {doc.metadata.get("relevance_score","N/A")}")
    print(doc.page_content[:200])
    print("---")

rele_score 0.64207345
한국종팔(전국 근무) 국내영업마켓팅
포지션 소개 Job Overview
삼성전자 DX부문에서 국내 시장에 출시하는 전 제품과 서비스의 국내 영업 마켓팅올 총관하는 직무입니다. 온 오프라인 유통 채널별 영업 전락 및 제품과 브랜드의 마켓팅 전락올 수립하다 경영 성과름 창출하고 있습니다.
B2CB2B온라인 경로의 판매 전락올 수립하고 신규 거래선과 Biz 틀 
---
rele_score 0.64207345
한국종팔(전국 근무) 국내영업마켓팅
포지션 소개 Job Overview
삼성전자 DX부문에서 국내 시장에 출시하는 전 제품과 서비스의 국내 영업 마켓팅올 총관하는 직무입니다. 온 오프라인 유통 채널별 영업 전락 및 제품과 브랜드의 마켓팅 전락올 수립하다 경영 성과름 창출하고 있습니다.
B2CB2B온라인 경로의 판매 전락올 수립하고 신규 거래선과 Biz 틀 
---
rele_score 0.64207345
영상디스플레이사업부(수원 근무) 해외영업 B2B 영업
포지션 소개 Job Overview
시장트랜드 센싱올 바탕으로 사업 전락올 분석 및 수립하고 다양한 영업활동올 통해 해외 업체들과의 파트너십올 구취관리하여 Tizen OS 라이센싱 및 아웃소상 사업흘 확장하고 새로운 사업기회틀 발굴합니다.
수행업무 Job Details
시장기술경쟁사 협력업체 환경 변화 
---
rele_score 0.64207345
영상디스플레이사업부(수원 근무) 해외영업 B2B 영업
포지션 소개 Job Overview
시장트랜드 센싱올 바탕으로 사업 전락올 분석 및 수립하고 다양한 영업활동올 통해 해외 업체들과의 파트너십올 구취관리하여 Tizen OS 라이센싱 및 아웃소상 사업흘 확장하고 새로운 사업기회틀 발굴합니다.
수행업무 Job Details
시장기술경쟁사 협력업체 환경 변화 
---
rele_score 0.640276
DA사업부(수원 근무) 해외영업 B2B 영업
포지션 소개 Job Overview
글로벌 시장과 파트너사에 대한 이해틀 바탕으로 생

#### LLMChainExtractor
- 질문 + 문서 같이 LLM에게 보내서 질문과 관련된 내용만 추출
- 현재 받은 chunks 안에 있는 내용을 줄여내는 것

In [30]:

# 기존꺼 호출
vectorstore = Chroma(collection_name="jeju1",persist_directory="./db/chroma_db", embedding_function=embeddings)
base_retriever = create_retriever(vector_store=vectorstore,k=20)
query = "제주도에서 먹어 볼만한 음식은?"

#base_retriever.invoke(query)


extractor = LLMChainExtractor.from_llm(llm=llm)
compression_retriever = ContextualCompressionRetriever(base_retriever=base_retriever,base_compressor=extractor)

docs = compression_retriever.invoke(query)
print(len(docs))
for doc in docs:
    print(doc.page_content[:200])
    print("---")



15
제주에서 꼭 맛봐야 할 음식
흑돼지
제주의 흑돼지 음식점은 어딜 가나 맛있으니까. 흑돼지 구이는 
멸치로 만든 젓갈인 ‘멜젓’ 에 찍어 먹으면 맛이 더욱 좋아진다. 
고기의 기름진 느끼함을 없애줌과 동시에 입맛을 돋운다. 
깨끗한 바다와 산이 둘러싸인 제주의 색다른 맛. 진정한 제주의 
음식을 접하고 싶다면 제주의 향토음식을 그냥 지나칠 수 없다. 
편견을
---
수제햄버거
간편해 보이지만 결코 가벼운 맛은 
아닌 수제햄버거 맛집도 많다. 제주 
로컬 재료들을 이용하거나 자기만의 
특별한 레시피로 승부하는 맛집들. 
바다를 바라보며 먹는 햄버거는 
제주에서 누릴 수 있는 특권일 것이다.
두루치기
고추장 양념된 돼지고기에 콩나물, 파채 등을 올려 볶아낸 제주 
두루치기. 흑돼지의 고장에서 먹는 두루치기는 육지의 맛과 
---
생선회
바다로 둘러싸인 제주에서 생선회는 흔한 음식이다. 
최고 횟감으로 불리는 다금바리부터 방어, 갈치, 고등어, 
한치까지 수많은 어종의 횟감이 즐비하다. 코스별로 
준비해 주는 횟집 말고도 시장이나 마트 코너에서도 
가성비 좋은 회를 즐길 수 있다.
물회
무더운 여름에 시원한 물회만 한 음식도 
드물다. 자리물회, 전복물회, 한치물회 등 
재료가 싱싱
---
약속한 고깃집들이 모여 있는 거리이다. 여기에서라면 제주도까지 와서 다른 지역 돼지고기를 먹을 걱정을 안 
해도 될 듯!
꿩요리                                                                                                    
제주엔 꿩이 많아 꿩요리 종류가 다양하다. 
꿩의 
---
김밥
김밥은 특별한 음식이 아니라고 할 분도 있겠다. 하지만 
제주에서만 맛볼 수 있는 특별한 김밥들도 있다. 수많은 
연예인들까지 인증한 예약을 하지 않으면 맛볼 수 없는 김밥부터 
한 줄을 다 먹기도 버거운  엄청난 크기의  김밥, 전복 한 마리가 
통 크게 올라간 김밥까지. 오늘 점심은 김밥도 좋겠다.
모닥치기

##### EmbeddingsFilter
- LLMChainExtractor 하기 전 사용하는 편. (임계값을 기반으로 문서 제외)


In [34]:
base_retriever = create_retriever(vectorstore, k=20)

embedding_filter=EmbeddingsFilter(embeddings=embeddings,similarity_threshold=0.6)

# embedding_filter => extractor(llmchain)
pipeline = DocumentCompressorPipeline(transformers=[embedding_filter, extractor])

compression_retriever = ContextualCompressionRetriever(base_compressor=pipeline, base_retriever=base_retriever)

print_retrieved_docs(title="EmbeddingsFilter + LLMChainExtractor",retriever=compression_retriever,query=query)



EmbeddingsFilter + LLMChainExtractor


In [ ]:
1